<a href="https://colab.research.google.com/github/josesabillon9597-ux/PROYECTO/blob/main/Clothing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#librerias
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
pd.options.display.float_format = '{:.2f}'.format
plt.figure(figsize=(15, 6))

<Figure size 1500x600 with 0 Axes>

<Figure size 1500x600 with 0 Axes>

In [ ]:
df=pd.read_csv('Multiclass Clothing Sales Dataset.csv')

In [ ]:
df.shape

(100000, 20)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Product_Category     100000 non-null  object 
 1   Brand                100000 non-null  object 
 2   Product_Name         100000 non-null  object 
 3   Gender               100000 non-null  object 
 4   Size                 100000 non-null  object 
 5   Color                100000 non-null  object 
 6   Season               100000 non-null  object 
 7   Payment_Method       100000 non-null  object 
 8   Customer_Type        100000 non-null  object 
 9   Selling_Price        95000 non-null   float64
 10  Cost_Price           100000 non-null  float64
 11  Discount_Percentage  95000 non-null   float64
 12  Quantity_Sold        100000 non-null  int64  
 13  Total_Sales          100000 non-null  float64
 14  Stock_Availability   95000 non-null   float64
 15  Customer_Age      

In [ ]:
df.isnull().sum().sort_values(ascending=False)

,0
Customer_Age,5000
Stock_Availability,5000
Selling_Price,5000
Discount_Percentage,5000
Store_Rating,5000
Product_Name,0
Brand,0
Product_Category,0
Gender,0
Payment_Method,0


In [ ]:
filtro_vacio_precio_venta=df['Selling_Price'].isnull()
df[filtro_vacio_precio_venta][['Selling_Price','Cost_Price']]

,Selling_Price,Cost_Price
38,NaN,982.33
77,NaN,499.88
79,NaN,980.82
101,NaN,1255.25
109,NaN,710.49
...,...,...
99917,NaN,11915.74
99932,NaN,6312.56
99945,NaN,4595.06
99949,NaN,4261.66


In [ ]:
df[['Cost_Price','Selling_Price','Quantity_Sold','Total_Sales']]

,Cost_Price,Selling_Price,Quantity_Sold,Total_Sales
0,816.03,1817.43,6,42241.69
1,793.24,1672.04,8,69318.54
2,890.47,1381.57,5,56851.12
3,1497.08,859.51,4,72084.36
4,896.19,1642.18,2,29846.08
...,...,...,...,...
99995,9140.00,10767.18,63,75646.22
99996,9703.81,19506.23,105,46527.21
99997,11436.59,9130.51,172,44611.20
99998,11531.87,19014.37,154,43867.12


Los ingresos no concuerdad con (costo de precio *cantidad vendida - descuento )

In [ ]:
#borrar la columna venta total ya que no tenia ningun sentidos sus digitos y la
#volvemos a crear
df_modificado = df.drop(columns=['Total_Sales'])

# 2. Creamos la columna real
df_modificado['Total_Sales_Real'] = (df['Selling_Price'] * df['Quantity_Sold']) * (1-(df['Discount_Percentage']/100))

# 3. Revision delos datos
df_modificado[[ 'Selling_Price', 'Quantity_Sold', 'Discount_Percentage' ,'Total_Sales_Real' ,'Cost_Price']]

,Selling_Price,Quantity_Sold,Discount_Percentage,Total_Sales_Real,Cost_Price
0,1817.43,6,26.26,8040.97,816.03
1,1672.04,8,26.88,9780.41,793.24
2,1381.57,5,30.27,4816.51,890.47
3,859.51,4,NaN,NaN,1497.08
4,1642.18,2,NaN,NaN,896.19
...,...,...,...,...,...
99995,10767.18,63,38.98,413916.28,9140.00
99996,19506.23,105,11.78,1806838.67,9703.81
99997,9130.51,172,25.18,1174933.48,11436.59
99998,19014.37,154,2.79,2846553.69,11531.87


In [ ]:
df_modificado.isnull().sum().sort_values(ascending=False)
#Al borrar y crear de nuevo la columna al  tener datos vacios en selling_price entonces nos dara celdas vacias
# la columna modificada

,0
Total_Sales_Real,9746
Selling_Price,5000
Discount_Percentage,5000
Stock_Availability,5000
Customer_Age,5000
Store_Rating,5000
Brand,0
Product_Category,0
Gender,0
Product_Name,0


hay 5mil dados vacios en el descuento , rellenaremos esas casillas segun las categorias  promediadas

In [ ]:
media_del_descuento=df['Discount_Percentage'].mean()
desv_del_desc=df['Discount_Percentage'].std()
print(media_del_descuento)
print(desv_del_desc)
coe_var=desv_del_desc/media_del_descuento
print(coe_var)


24.99438861981896
14.431100365845415
0.5773736091469135


un coeficiente de variacion muy alta 57% hay mucha variedad entre los descuentos  ,analizamos los  cuantiles


In [ ]:
df['Discount_Percentage'].describe()

,Discount_Percentage
count,95000.00
mean,24.99
std,14.43
min,0.00
25%,12.45
50%,25.01
75%,37.45
max,50.00


notamos que la media en si solo representa el 25% anda al rededor de 12.45 pero de 25.01 que representa el 50% de los  descuentos anda  casi eldoble  asi que usaremos la mediana para eleminar ese sesgos

In [ ]:
mediana_del_descuento=df['Discount_Percentage'].median()
print(mediana_del_descuento)

25.007454061924577


ya la mediana representa un numero porcentaje de descuento que  su maximo para arriba es un 50 % pero para abajo tambien un 50% donde los datos se ajusta ente si , rellenamos las celdas con la mediana

mientra que  que la media representaba mas del 70 % en comparacion al maximos en un 25% de los datos  (muy  sesgado  )

In [ ]:
# rellenar por la mediana las casillas vacias  de los descuentos
df_modificado['Discount_Percentage']=df_modificado['Discount_Percentage'].fillna(mediana_del_descuento)
df_modificado

,Product_Category,Brand,Product_Name,Gender,Size,Color,Season,Payment_Method,Customer_Type,Selling_Price,Cost_Price,Discount_Percentage,Quantity_Sold,Stock_Availability,Customer_Age,Purchase_Frequency,Store_Rating,Return_Rate,Sales_Category,Total_Sales_Real
0,Traditional Wear,Forever 21,Tops,Women,S,White,Winter,Card,New,1817.43,816.03,26.26,6,294.00,58.00,2.56,3.11,29.29,High Sales,8040.97
1,Athleisure,Ralph Lauren,Casual Shirt,Men,XXL,Yellow,All-Season,Card,Returning,1672.04,793.24,26.88,8,376.00,40.00,1.59,3.43,19.47,Low Sales,9780.41
2,Outerwear,Nike,Blazer,Men,S,Green,All-Season,UPI,Returning,1381.57,890.47,30.27,5,48.00,41.00,3.07,3.77,22.08,Medium Sales,4816.51
3,Athleisure,Zara,Leggings,Women,M,White,Winter,UPI,Returning,859.51,1497.08,25.01,4,362.00,32.00,2.19,4.53,17.28,Medium Sales,NaN
4,Athleisure,Zara,Dress,Men,XL,Yellow,Summer,Net Banking,Returning,1642.18,896.19,25.01,2,34.00,27.00,2.01,4.64,9.38,Medium Sales,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,Bottoms,Reebok,Shorts,Women,M,Blue,Winter,Card,Returning,10767.18,9140.00,38.98,63,NaN,60.00,29.00,1.19,15.57,Low Sales,413916.28
99996,Tops,Zara,Shorts,Women,L,Blue,All-Season,Net Banking,New,19506.23,9703.81,11.78,105,390.00,28.00,36.00,1.65,22.95,Medium Sales,1806838.67
99997,Athleisure,Calvin Klein,Jacket,Women,XL,Orange,Winter,UPI,Returning,9130.51,11436.59,25.18,172,435.00,61.00,23.00,1.43,15.80,Low Sales,1174933.48
99998,Traditional Wear,Nike,Sweatshirt,Women,S,White,Winter,Cash,New,19014.37,11531.87,2.79,154,452.00,22.00,46.00,2.14,1.87,High Sales,2846553.69


In [ ]:
df_modificado.isnull().sum().sort_values(ascending=False) #revisar

,0
Total_Sales_Real,9746
Store_Rating,5000
Selling_Price,5000
Stock_Availability,5000
Customer_Age,5000
Product_Name,0
Brand,0
Product_Category,0
Gender,0
Size,0


In [ ]:
df_modificado

,Product_Category,Brand,Product_Name,Gender,Size,Color,Season,Payment_Method,Customer_Type,Selling_Price,Cost_Price,Discount_Percentage,Quantity_Sold,Stock_Availability,Customer_Age,Purchase_Frequency,Store_Rating,Return_Rate,Sales_Category,Total_Sales_Real
0,Traditional Wear,Forever 21,Tops,Women,S,White,Winter,Card,New,1817.43,816.03,26.26,6,294.00,58.00,2.56,3.11,29.29,High Sales,8040.97
1,Athleisure,Ralph Lauren,Casual Shirt,Men,XXL,Yellow,All-Season,Card,Returning,1672.04,793.24,26.88,8,376.00,40.00,1.59,3.43,19.47,Low Sales,9780.41
2,Outerwear,Nike,Blazer,Men,S,Green,All-Season,UPI,Returning,1381.57,890.47,30.27,5,48.00,41.00,3.07,3.77,22.08,Medium Sales,4816.51
3,Athleisure,Zara,Leggings,Women,M,White,Winter,UPI,Returning,859.51,1497.08,25.01,4,362.00,32.00,2.19,4.53,17.28,Medium Sales,NaN
4,Athleisure,Zara,Dress,Men,XL,Yellow,Summer,Net Banking,Returning,1642.18,896.19,25.01,2,34.00,27.00,2.01,4.64,9.38,Medium Sales,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,Bottoms,Reebok,Shorts,Women,M,Blue,Winter,Card,Returning,10767.18,9140.00,38.98,63,NaN,60.00,29.00,1.19,15.57,Low Sales,413916.28
99996,Tops,Zara,Shorts,Women,L,Blue,All-Season,Net Banking,New,19506.23,9703.81,11.78,105,390.00,28.00,36.00,1.65,22.95,Medium Sales,1806838.67
99997,Athleisure,Calvin Klein,Jacket,Women,XL,Orange,Winter,UPI,Returning,9130.51,11436.59,25.18,172,435.00,61.00,23.00,1.43,15.80,Low Sales,1174933.48
99998,Traditional Wear,Nike,Sweatshirt,Women,S,White,Winter,Cash,New,19014.37,11531.87,2.79,154,452.00,22.00,46.00,2.14,1.87,High Sales,2846553.69


Rellenar las casillas vacias de la columna precio de venta
realizar el analisis  correspondiente

La columna  del rating se rellena con el promedio ademas esata columna no  nos ayuda mucho en el analisis  y mas si es por esca de numero eje esacal del 1 al 5

In [ ]:
df_modificado['Store_Rating'].describe() # aqui vemos que la mediana y media es lamisma
df_modificado['Store_Rating']=df_modificado['Store_Rating'].fillna(df_modificado['Store_Rating'].mean())
df_modificado.isnull().sum().sort_values(ascending=False)

,0
Total_Sales_Real,9746
Customer_Age,5000
Selling_Price,5000
Stock_Availability,5000
Gender,0
Product_Name,0
Brand,0
Product_Category,0
Payment_Method,0
Size,0


ANALISAMOS LOS PRECIOS DE VENTA Y LOS RELLENAMOS CON  LA MEDIANA O MEDIANA SEGUN CONVENGA O SEGUN LA CATEGORIA

In [ ]:
df_modificado['Selling_Price'].describe() #media y  mediana  muy parecida
# encontramos que tenemos datos negativo  eso no es posible  borrramos  y rellenamos

,Selling_Price
count,95000.00
mean,1554.03
std,974.71
min,-613.62
25%,1162.14
50%,1501.41
75%,1842.75
max,19989.22


In [ ]:

df_modificado['Selling_Price'] = df_modificado['Selling_Price'].clip(lower=0)
df_modificado['Selling_Price'].describe() # la mediana y la media nocambio mucho asi que ultimo recurso usamos
# la media por grupos

,Selling_Price
count,95000.00
mean,1554.21
std,974.40
min,0.00
25%,1162.14
50%,1501.41
75%,1842.75
max,19989.22


In [ ]:
df_modificado['Selling_Price']=df_modificado['Selling_Price'].fillna(df_modificado.groupby('Product_Category')['Selling_Price'].transform('median'))
df_modificado.isnull().sum().sort_values(ascending=False) #ahora si rellenamos  la columna  total seale real

,0
Total_Sales_Real,9746
Stock_Availability,5000
Customer_Age,5000
Product_Category,0
Gender,0
Product_Name,0
Brand,0
Size,0
Customer_Type,0
Color,0


In [ ]:
df_modificado.isnull().sum().sort_values(ascending=False)

,0
Total_Sales_Real,9746
Stock_Availability,5000
Customer_Age,5000
Product_Category,0
Gender,0
Product_Name,0
Brand,0
Size,0
Customer_Type,0
Color,0


In [ ]:
df_modificado['Total_Sales_Real'] = df_modificado['Total_Sales_Real'].fillna(
    (df_modificado['Selling_Price'] * df_modificado['Quantity_Sold']) * (1 - (df_modificado['Discount_Percentage'] / 100))
)
df_modificado

,Product_Category,Brand,Product_Name,Gender,Size,Color,Season,Payment_Method,Customer_Type,Selling_Price,Cost_Price,Discount_Percentage,Quantity_Sold,Stock_Availability,Customer_Age,Purchase_Frequency,Store_Rating,Return_Rate,Sales_Category,Total_Sales_Real
0,Traditional Wear,Forever 21,Tops,Women,S,White,Winter,Card,New,1817.43,816.03,26.26,6,294.00,58.00,2.56,3.11,29.29,High Sales,8040.97
1,Athleisure,Ralph Lauren,Casual Shirt,Men,XXL,Yellow,All-Season,Card,Returning,1672.04,793.24,26.88,8,376.00,40.00,1.59,3.43,19.47,Low Sales,9780.41
2,Outerwear,Nike,Blazer,Men,S,Green,All-Season,UPI,Returning,1381.57,890.47,30.27,5,48.00,41.00,3.07,3.77,22.08,Medium Sales,4816.51
3,Athleisure,Zara,Leggings,Women,M,White,Winter,UPI,Returning,859.51,1497.08,25.01,4,362.00,32.00,2.19,4.53,17.28,Medium Sales,2578.27
4,Athleisure,Zara,Dress,Men,XL,Yellow,Summer,Net Banking,Returning,1642.18,896.19,25.01,2,34.00,27.00,2.01,4.64,9.38,Medium Sales,2463.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,Bottoms,Reebok,Shorts,Women,M,Blue,Winter,Card,Returning,10767.18,9140.00,38.98,63,NaN,60.00,29.00,1.19,15.57,Low Sales,413916.28
99996,Tops,Zara,Shorts,Women,L,Blue,All-Season,Net Banking,New,19506.23,9703.81,11.78,105,390.00,28.00,36.00,1.65,22.95,Medium Sales,1806838.67
99997,Athleisure,Calvin Klein,Jacket,Women,XL,Orange,Winter,UPI,Returning,9130.51,11436.59,25.18,172,435.00,61.00,23.00,1.43,15.80,Low Sales,1174933.48
99998,Traditional Wear,Nike,Sweatshirt,Women,S,White,Winter,Cash,New,19014.37,11531.87,2.79,154,452.00,22.00,46.00,2.14,1.87,High Sales,2846553.69


In [ ]:
# 1.costo total de las piezas vendidas
costo_total_operacion = df_modificado['Cost_Price'] * df_modificado['Quantity_Sold']
df_modificado['Ganancia_Neta'] = df_modificado['Total_Sales_Real'] - costo_total_operacion

# 3.
print("Ganancia total de la tienda: $", df_modificado['Ganancia_Neta'].sum(),
        "costo de opercion es $",costo_total_operacion.sum())


Ganancia total de la tienda: $ 53809164.91634188 costo de opercion es $ 841177197.2873578


In [ ]:
df_modificado[['Selling_Price','Quantity_Sold','Total_Sales_Real']]

,Selling_Price,Quantity_Sold,Total_Sales_Real
0,1817.43,6,8040.97
1,1672.04,8,9780.41
2,1381.57,5,4816.51
3,859.51,4,2578.27
4,1642.18,2,2463.02
...,...,...,...
99995,10767.18,63,413916.28
99996,19506.23,105,1806838.67
99997,9130.51,172,1174933.48
99998,19014.37,154,2846553.69


In [ ]:
df_modificado.isnull().sum().sort_values(ascending=False)

,0
Customer_Age,5000
Stock_Availability,5000
Product_Name,0
Brand,0
Product_Category,0
Size,0
Gender,0
Color,0
Season,0
Selling_Price,0


Rellenamos las casillas vacias del rating con el promedio o mediana
aqui no afecta mucho ya  que son rating segun rangos

In [ ]:
df_modificado['Store_Rating'].describe() #aque vemos que enefecto la mediana y la media son iguales ya que
 #su medicion es por rango corto vemos claramente que su maximo es 5 estamos solo a un 20%  de la media

,Store_Rating
count,95000.00
mean,4.00
std,0.59
min,1.00
25%,3.50
50%,4.00
75%,4.50
max,5.00


In [ ]:
df_modificado['Store_Rating']=df_modificado['Store_Rating'].fillna(df_modificado['Store_Rating'].mean())
df_modificado.isnull().sum().sort_values(ascending=False)

,0
Customer_Age,5000
Stock_Availability,5000
Product_Name,0
Brand,0
Product_Category,0
Size,0
Gender,0
Color,0
Season,0
Selling_Price,0


In [ ]:
df_modificado

,Product_Category,Brand,Product_Name,Gender,Size,Color,Season,Payment_Method,Customer_Type,Selling_Price,...,Discount_Percentage,Quantity_Sold,Stock_Availability,Customer_Age,Purchase_Frequency,Store_Rating,Return_Rate,Sales_Category,Total_Sales_Real,Ganancia_Neta
0,Traditional Wear,Forever 21,Tops,Women,S,White,Winter,Card,New,1817.43,...,26.26,6,294.00,58.00,2.56,3.11,29.29,High Sales,8040.97,3144.80
1,Athleisure,Ralph Lauren,Casual Shirt,Men,XXL,Yellow,All-Season,Card,Returning,1672.04,...,26.88,8,376.00,40.00,1.59,3.43,19.47,Low Sales,9780.41,3434.51
2,Outerwear,Nike,Blazer,Men,S,Green,All-Season,UPI,Returning,1381.57,...,30.27,5,48.00,41.00,3.07,3.77,22.08,Medium Sales,4816.51,364.16
3,Athleisure,Zara,Leggings,Women,M,White,Winter,UPI,Returning,859.51,...,25.01,4,362.00,32.00,2.19,4.53,17.28,Medium Sales,2578.27,-3410.05
4,Athleisure,Zara,Dress,Men,XL,Yellow,Summer,Net Banking,Returning,1642.18,...,25.01,2,34.00,27.00,2.01,4.64,9.38,Medium Sales,2463.02,670.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,Bottoms,Reebok,Shorts,Women,M,Blue,Winter,Card,Returning,10767.18,...,38.98,63,NaN,60.00,29.00,1.19,15.57,Low Sales,413916.28,-161903.71
99996,Tops,Zara,Shorts,Women,L,Blue,All-Season,Net Banking,New,19506.23,...,11.78,105,390.00,28.00,36.00,1.65,22.95,Medium Sales,1806838.67,787938.63
99997,Athleisure,Calvin Klein,Jacket,Women,XL,Orange,Winter,UPI,Returning,9130.51,...,25.18,172,435.00,61.00,23.00,1.43,15.80,Low Sales,1174933.48,-792160.61
99998,Traditional Wear,Nike,Sweatshirt,Women,S,White,Winter,Cash,New,19014.37,...,2.79,154,452.00,22.00,46.00,2.14,1.87,High Sales,2846553.69,1070646.34


De  igual manera rellenamos la edad con el promedio o  mejor con la moda

In [ ]:
df_modificado['Customer_Age'].describe()

,Customer_Age
count,95000.00
mean,41.07
std,13.57
min,18.00
25%,29.00
50%,41.00
75%,53.00
max,64.00


In [ ]:
# cantidad exacta de veces que se repite la edad más común(moda)
veces_que_se_repite = df_modificado['Customer_Age'].value_counts().iloc[0]
print("La moda se repite:", veces_que_se_repite, "veces")  #podemos usar cualquiera de las dos son muy parecidas
#usaremos la moda


La moda se repite: 2128 veces


In [ ]:
df_modificado['Customer_Age']=df_modificado['Customer_Age'].fillna(df_modificado['Customer_Age'].mode()[0])
df_modificado.isnull().sum().sort_values(ascending=False)

,0
Stock_Availability,5000
Brand,0
Product_Name,0
Gender,0
Product_Category,0
Size,0
Color,0
Payment_Method,0
Season,0
Selling_Price,0


In [ ]:
df_modificado

,Product_Category,Brand,Product_Name,Gender,Size,Color,Season,Payment_Method,Customer_Type,Selling_Price,...,Discount_Percentage,Quantity_Sold,Stock_Availability,Customer_Age,Purchase_Frequency,Store_Rating,Return_Rate,Sales_Category,Total_Sales_Real,Ganancia_Neta
0,Traditional Wear,Forever 21,Tops,Women,S,White,Winter,Card,New,1817.43,...,26.26,6,294.00,58.00,2.56,3.11,29.29,High Sales,8040.97,3144.80
1,Athleisure,Ralph Lauren,Casual Shirt,Men,XXL,Yellow,All-Season,Card,Returning,1672.04,...,26.88,8,376.00,40.00,1.59,3.43,19.47,Low Sales,9780.41,3434.51
2,Outerwear,Nike,Blazer,Men,S,Green,All-Season,UPI,Returning,1381.57,...,30.27,5,48.00,41.00,3.07,3.77,22.08,Medium Sales,4816.51,364.16
3,Athleisure,Zara,Leggings,Women,M,White,Winter,UPI,Returning,859.51,...,25.01,4,362.00,32.00,2.19,4.53,17.28,Medium Sales,2578.27,-3410.05
4,Athleisure,Zara,Dress,Men,XL,Yellow,Summer,Net Banking,Returning,1642.18,...,25.01,2,34.00,27.00,2.01,4.64,9.38,Medium Sales,2463.02,670.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,Bottoms,Reebok,Shorts,Women,M,Blue,Winter,Card,Returning,10767.18,...,38.98,63,NaN,60.00,29.00,1.19,15.57,Low Sales,413916.28,-161903.71
99996,Tops,Zara,Shorts,Women,L,Blue,All-Season,Net Banking,New,19506.23,...,11.78,105,390.00,28.00,36.00,1.65,22.95,Medium Sales,1806838.67,787938.63
99997,Athleisure,Calvin Klein,Jacket,Women,XL,Orange,Winter,UPI,Returning,9130.51,...,25.18,172,435.00,61.00,23.00,1.43,15.80,Low Sales,1174933.48,-792160.61
99998,Traditional Wear,Nike,Sweatshirt,Women,S,White,Winter,Cash,New,19014.37,...,2.79,154,452.00,22.00,46.00,2.14,1.87,High Sales,2846553.69,1070646.34


In [ ]:
df_modificado.isnull().sum().sort_values(ascending=False)

,0
Stock_Availability,5000
Brand,0
Product_Name,0
Gender,0
Product_Category,0
Size,0
Color,0
Payment_Method,0
Season,0
Selling_Price,0


In [ ]:
grafico_de_productos=df_modificado['Product_Name'].value_counts().sort_values(ascending=True)

grafico_de_productos.plot(marker='o')

#rotacion de los nombre quede vertical
plt.xticks(range(len(grafico_de_productos)), grafico_de_productos.index, rotation=90)

# 5. Agregar títulos organizados
plt.title("Cantidad de veces que se repite cada producto (Menor a Mayor)")
plt.xlabel("Nombre del Producto")
plt.ylabel("Veces Repetido")
plt.grid(True, linestyle='--', alpha=0.5) # Líneas de fondo opacas

# 6. Mostrar el resultado final sin recortes
plt.tight_layout()
plt.show()

NameError: name 'df_modificado' is not defined